# Crystal Eye | ACME Hands-on in Python

**Crystal Eye tutorial team · ACME Hands-on**

- **Generate:** choose a detector-frame direction and inspect the expected counts per pixel.
- **Detect:** estimate the background from the first 8 seconds and look for a burst in the residuals.
- **Localize:** compare the pixel pattern with response templates and measure the angular separation.

Run the cells from top to bottom. The `crystal_eye_acme.py` uses PyROOT to read the supplied detector simulations and evaluate the ROOT statistical operations.

For downloads and environment setup, see the [exercise instructions](../hands-on.md).

In [ ]:
import os
from pathlib import Path
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from crystal_eye_acme import generate_grb, lightcurve, localize

source = Path(os.environ.get("ACME_TUTORIAL_DIR", "../../ACMETutorial")).expanduser().resolve()
required = ("allBackground.root", "allEffArea.root", "templates5deg.root",
            "temEdepPix2degNew.root", "upCal.txt")
missing = [name for name in required if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f"Set ACME_TUTORIAL_DIR to the shared input directory. Missing: {missing}")

from matplotlib.ticker import MaxNLocator, StrMethodFormatter

INK, MUTED = "#111827", "#4B5563"
TEAL, BLUE, AMBER = "#0F766E", "#1D4ED8", "#B45309"
BASELINE, GRID = "#6B7280", "#D1D5DB"

plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 300,
    "figure.facecolor": "white", "axes.facecolor": "white",
    "font.family": "DejaVu Sans", "font.size": 10,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.titlecolor": INK,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#9CA3AF", "axes.linewidth": 0.8,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "legend.frameon": False,
})

def figure_header(fig, title, subtitle, note):
    fig.text(.10, .95, title, ha="left", va="top", fontsize=15, weight="bold")
    fig.text(.10, .885, subtitle, ha="left", va="top", color=MUTED, fontsize=10)
    fig.text(.10, .035, note, ha="left", va="bottom", color="#6B7280", fontsize=8.5)

def style_counts(ax):
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", color=GRID, alpha=.55)
    ax.yaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))

def show_plot(fig):
    png = BytesIO()
    fig.savefig(png, format="png", dpi=150, bbox_inches="tight", pad_inches=.15)
    plt.close(fig)
    display(Image(data=png.getvalue()))


## Detector inputs

The notebook uses the supplied Geant4 reference products. The [detector chapter](../theory/detector.md) describes the geometry, and the [response chapter](../theory/response.md) explains how the template banks were built.

## 1. Generate a burst

Choose detector-frame zenith and azimuth angles in degrees, and the parameters of the cut-off power law. The code selects neighboring response templates and estimates the energy spectrum and counts per pixel. First run the supplied example. Then try an angle of 45° or reduce the amplitude by a factor of ten.

The first plot shows how the expected source count rate varies across detector pixels for your settings.

In [ ]:
theta_deg, phi_deg = 0, 0
amplitude, spectral_index, cutoff_kev = 0.026, -1.07, 756.4
response = generate_grb(source, theta_deg, phi_deg, amplitude, spectral_index, cutoff_kev)
print(f"Expected total detector rate: {response.energy.Integral():.2f} counts/s")
for theta, phi, weight, events in response.templates:
    print(f"Template θ={theta:.2f}°, φ={phi:.2f}° · weight {weight:.3f} · {events:,} events")

fig, ax = plt.subplots(figsize=(9.2, 4.0))
fig.subplots_adjust(left=.10, right=.975, top=.79, bottom=.22)
figure_header(
    fig, "The expected burst response varies across pixels",
    f"Detector direction: θ = {theta_deg:g}°, φ = {phi_deg:g}° · expected source counts per second",
    "Source: Crystal Eye generator templates · simulated source contribution.",
)
pixel = range(response.pixels.GetNbinsX())
counts_per_pixel = [response.pixels.GetBinContent(i + 1) for i in pixel]
ax.step(pixel, counts_per_pixel, where="mid", color=TEAL, linewidth=1.45)
ax.set(xlabel="Detector pixel ID", ylabel="Expected counts/s", xlim=(0, len(pixel) - 1))
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
style_counts(ax)
show_plot(fig)


## 2. Detection: when did the counts rise?
The Python port of **CELightcurve.cc** generates 250 ms count bins with Gaussian fluctuations. The top plot compares simulated total counts to the model background mean and its counting spread. The lower plot subtracts the mean estimated from the first 8 s. Look for an excess before revealing the expected injected signal. Change the amplitude in section 1, then rerun sections 1 and 2 with the same seed.

The program also reports its original experimental trigger. Its threshold uses the dispersion of the 8 s aggregate and compares it with a single-bin excess. Treat it as program output rather than a calibrated significance.

In [ ]:
seed = 2026
time_s, total, background, injected_signal, triggers = lightcurve(source, response, seed)
print(f"Legacy trigger onsets (s): {triggers or 'none'}")

In [ ]:
reveal_injected = False

baseline_window = time_s < 8
estimated_background = total[baseline_window].mean()
fig, (ax, residual) = plt.subplots(2, 1, sharex=True, figsize=(10, 6.7))
fig.subplots_adjust(left=.10, right=.975, top=.79, bottom=.16, hspace=.37)
figure_header(
    fig, "Does an excess stand out above the background?",
    "30-second simulation · 250 ms bins · background estimated from the first 8 seconds",
    "Source: Crystal Eye simulated light curve · shaded spread is ±√mean, not a trigger threshold.",
)
ax.step(time_s, total, where="mid", color=BLUE, linewidth=1.35)
ax.axhline(background, color=BASELINE, linestyle="--", linewidth=1)
ax.fill_between(time_s, max(0, background - np.sqrt(background)),
                background + np.sqrt(background), color=GRID, alpha=.48)
ax.set_title("Total counts and expected background", loc="left", fontsize=11, weight="bold")
ax.set_ylabel("Counts/bin")
residual.step(time_s, total - estimated_background, where="mid", color=TEAL, linewidth=1.35)
residual.axhline(0, color=BASELINE, linewidth=.8)
residual.axvspan(0, 8, color=GRID, alpha=.26)
if reveal_injected:
    residual.step(time_s, injected_signal, where="mid", color=AMBER, linestyle="--", linewidth=1.55)
residual.set_title(
    "Counts above or below the estimated mean" + (" · dashed: injected GRB" if reveal_injected else ""),
    loc="left", fontsize=11, weight="bold",
)
residual.set(xlabel="Time (s)", ylabel="Excess counts/bin", xlim=(0, 30))
for chart in (ax, residual):
    style_counts(chart)
show_plot(fig)


## 3. Localization: where did the burst come from?

Use the expected source pixel counts to compute a detector centroid. The centroid selects an effective area. The code compares the pixel distributions to the direction templates with ROOT's Kolmogorov–Smirnov histogram test, then averages retained directions with the KS scores as weights.

This exercise uses the simulated source response. The noisy light curve is a separate detection exercise. The scores compare templates and do not define a confidence region.

The polar plot shows the injected and reconstructed directions alongside the retained templates. Radius is detector zenith and angle around the plot is detector azimuth.

In [ ]:
result = localize(source, response)
recovered_theta, recovered_phi = result.theta, result.phi

def unit_vector(theta, phi):
    t, p = np.deg2rad([theta, phi])
    return np.array([np.sin(t) * np.cos(p), np.sin(t) * np.sin(p), np.cos(t)])

injected = unit_vector(theta_deg, phi_deg)
recovered = unit_vector(recovered_theta, recovered_phi)
separation = np.rad2deg(np.arctan2(np.linalg.norm(np.cross(injected, recovered)), injected @ recovered))
print(f"Centroid θ={result.centroid_theta:.2f}°, φ={result.centroid_phi:.2f}°")
print(f"Reconstructed θ={recovered_theta:.2f}°, φ={recovered_phi:.2f}°")
print(f"Angular separation: {separation:.2f}°")
print(f"Retained directions: {result.retained.sum()} of {len(result.scores)}")

fig, ax = plt.subplots(subplot_kw={"projection": "polar"}, figsize=(8, 6.3))
fig.subplots_adjust(left=.20, right=.80, top=.78, bottom=.21)
figure_header(
    fig, f"The reconstructed direction is {separation:.2f}° from the injection",
    "Detector coordinates · radial distance = zenith · polar angle = azimuth",
    "Source: Crystal Eye detector response templates · markers do not define a confidence region.",
)
t, p, score = result.scores[result.retained].T
ax.scatter(np.deg2rad(p), t, s=16 + 62 * score / score.max(), color=TEAL,
           alpha=.45, label="Retained templates")
ax.scatter(np.deg2rad(phi_deg), theta_deg, s=95, facecolors="none",
           edgecolors=INK, linewidths=1.4, label="Injected")
ax.scatter(np.deg2rad(recovered_phi), recovered_theta, marker="+", s=125,
           color=AMBER, linewidths=2, label="Reconstructed")
ax.set_rmax(max(15, theta_deg + 5, recovered_theta + 5, float(t.max()) + 5))
ax.set_rlabel_position(27)
ax.yaxis.set_major_formatter(StrMethodFormatter("{x:.0f}°"))
ax.grid(color=GRID, alpha=.75, linewidth=.8)
ax.spines["polar"].set_color("#9CA3AF")
fig.legend(*ax.get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(.5, .09),
           ncol=3, fontsize=9)
show_plot(fig)


## Questions to try
1. How far can you reduce the amplitude before you can no longer identify the burst by eye? Does the program's legacy trigger agree?
2. Does reducing amplitude have the same effect as changing the direction? Explain what data localization receives.
3. How close is the weighted direction to the detector centroid? Compare both with the injected direction for an off-axis example.

The preparation of the Geant4-derived inputs is documented separately in [the theory chapters](../theory/detector.md).